In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)
import importlib
import os
import sys
#root_path = os.path.dirname(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)
from env.parameters import P
from analysis_functions.data_preparation import cohort_type_adjustment
import dask.dataframe as dd
import pickle
import yaml
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2


In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
df_trn_val = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/f_ml_prv_asthma_1_fold/f1_gphesonly_trn_val_df.csv''')

In [ ]:
df_trn_val["split"] = "trn_val"

In [ ]:
df_tst = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/f_ml_prv_asthma_1_fold/f1_gphesonly_tst_df.csv''')

In [ ]:
df_tst["split"] = "tst"

In [ ]:
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
print(df_trn_val.shape)



In [ ]:
# step 1
df_trn_val = cohort_type_adjustment(df_trn_val, cols_dict)
df_tst = cohort_type_adjustment(df_tst, cols_dict)




In [ ]:
df_in = pd.concat([df_trn_val, df_tst])

In [ ]:
from pipeline_functions import *
from custom_plots import *

In [ ]:
df_in["country_imd"].value_counts(dropna=False)

In [ ]:
df_in.shape

In [ ]:
df_in['flag_asthma_at_cohort_start'].value_counts()

In [ ]:
# Check cohort start date and compare to GP (which is available up to 2016 and 2017

df_in["evdt_cohort_start"].min()

In [ ]:
df_in["evdt_cohort_start"].max()

# Missing

In [ ]:
df_in[df_in["bmi_field"].isnull()].shape[0]

In [ ]:
df_in[df_in["bmi_field"].isnull()].shape[0]/df_in.shape[0]*100

In [ ]:
cols_dict.get("cols_float")

In [ ]:
df_in["flag_asthma_at_cohort_start"].value_counts()

In [ ]:
df_in[df_in["distance_major_road_field"].isnull()].shape[0]

In [ ]:
df_in[df_in["distance_nearest_road_field"].isnull()].shape[0]

In [ ]:
df_in[df_in["traffic_intensity_field"].isnull()].shape[0]

In [ ]:
df_in[df_in["pef_field"].isnull()].shape[0]

In [ ]:
df_in["country_imd"].value_counts()

In [ ]:
df_en = df_in[df_in["country_imd"]=="England"]
df_sct = df_in[df_in["country_imd"]=="Scotland"]
df_wl = df_in[df_in["country_imd"]=="Wales"]
df_unkn = df_in[df_in["country_imd"]=="Unknown"]


In [ ]:
df_unkn[df_unkn["traffic_intensity_field"].isnull()].shape[0]

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


def internal_impute_nulls_mice(df: pd.DataFrame, cols_for_imputation: list, 
                      cols_other: list, create_new_col: True) -> pd.DataFrame:
    """ A funciton to apply MICE multiple imputaion from the Fancy imputer

    Args:
        df: The entire dataframe
        cols_for_imputation: The list of columns that need imputation. These column must be numericalz
        cols_other: The list of columns used for multiple imputation
        create_new_cols: Whether to create new columns for imputed columns or to overwrite them.

    Returns:
        A dataframe with new columns prefixed with _imputad
    """
    all_cols = cols_for_imputation+cols_other
    #cols_for_imputation = ['bmi_field']
    mice_imputer = IterativeImputer(estimator=RandomForestRegressor(n_estimators=5), 
                                    max_iter=50, tol= 1e-3,
                                   random_state=27)
    imputed_data = mice_imputer.fit_transform(df[all_cols])
    imputed_df = pd.DataFrame(imputed_data, columns=all_cols)
    postfix_string = '_imputed' if create_new_col else ''
    for col in cols_for_imputation:
        df[f'{col}{postfix_string}'] = np.where(df[col].isnull(), imputed_df[col], df[col])

    return df.copy()

In [ ]:
def internal_missing_pipeline_maker(cols_health, cols_environment,  create_new_col=True):
    

    #cols_final = [x for x in cols_for_imputation_health if x in list_cols]
    cols_for_imputation_health = cols_health
    
    cols_other_health = ['age_cohort_start', 'sex_code', 'follow_up_asthma_pre_cohort_start',
                         'pheno_diabetes_pre_cohort_start',
                         'pheno_ckd_pre_cohort_start', 'pheno_cvd_pre_cohort_start', 
                         'pheno_ht_pre_cohort_start', 'pheno_copd_pre_cohort_start']
    imputer_health_transformer = FunctionTransformer(func=internal_impute_nulls_mice, 
                                                     kw_args={'cols_for_imputation': cols_for_imputation_health, 
                                                                                 'cols_other': cols_other_health,
                                                             'create_new_col':create_new_col})
    cols_for_imputation_environment = cols_environment
    cols_other_environment = ['imd']

    imputer_environment_transformer = FunctionTransformer(func=internal_impute_nulls_mice, 
                                                          kw_args={'cols_for_imputation': cols_for_imputation_environment, 
                                                                                 'cols_other': cols_other_environment,
                                                                  'create_new_col':create_new_col})
    pipeline_imputer = Pipeline([
    ('imputer_health', imputer_health_transformer),
    ('imputer_environment', imputer_environment_transformer)
    ])
    return pipeline_imputer




In [ ]:


def internal_missing_pipeline_maker_bmi_only(cols_health,  create_new_col=True):
    

    #cols_final = [x for x in cols_for_imputation_health if x in list_cols]
    cols_for_imputation_health = cols_health
    
    cols_other_health = ['age_cohort_start', 'sex_code', 'follow_up_asthma_pre_cohort_start',
                         'pheno_diabetes_pre_cohort_start',
                         'pheno_ckd_pre_cohort_start', 'pheno_cvd_pre_cohort_start', 
                         'pheno_ht_pre_cohort_start', 'pheno_copd_pre_cohort_start']
    imputer_health_transformer = FunctionTransformer(func=internal_impute_nulls_mice, 
                                                     kw_args={'cols_for_imputation': cols_for_imputation_health, 
                                                                                 'cols_other': cols_other_health,
                                                             'create_new_col':create_new_col})
   

    pipeline_imputer = Pipeline([
    ('imputer_health', imputer_health_transformer),
    ])
    return pipeline_imputer




In [ ]:
# if create_new_col, the columns will get _imputer at the end
cols_health = ['bmi_field']
cols_environment = ['distance_major_road_field', 'distance_nearest_road_field', 
                                       'traffic_intensity_field']

missing_pipeline = Pipeline([
    ('missing_imputer', internal_missing_pipeline_maker(cols_health= cols_health,
                                               cols_environment = cols_environment
                                               , create_new_col=True)),
])


feature_preprocess_pipeline = Pipeline([
   # ('outlier_pipeline', outlier_pipeline),
   ('missing_pipeline', missing_pipeline),
 #  ('discritiser_pipeline', discritiser_pipeline),
 #  ('feature_selection', feature_selection_pipeline)
])

Imputation of environmental columns are based on IMD. Do seperately for each country. For Unkown country, use median (where IMD is not available)

# Scotland

In [ ]:
df_sct[df_sct["bmi_field"].isnull()].shape[0]/df_sct.shape[0] *100

In [ ]:
df_sct[df_sct["bmi_field"]>=30].shape[0]/df_sct.shape[0]*100

In [ ]:
feature_preprocess_pipeline.fit(df_sct)
df_sct = feature_preprocess_pipeline.transform(df_sct)

In [ ]:
df_sct[df_sct["bmi_field_imputed"].isnull()].shape[0]

In [ ]:
df_sct[df_sct["bmi_field_imputed"]>=30].shape[0]/df_sct.shape[0]*100

In [ ]:
df_sct[(df_sct["bmi_field"].isnull())&(df_sct["bmi_field_imputed"]>=30)].shape[0]/df_sct.shape[0]*100

# Wales

In [ ]:
df_wl[df_wl["bmi_field"].isnull()].shape[0]/df_wl.shape[0] * 100

In [ ]:
df_wl[df_wl["bmi_field"]>=30].shape[0]/df_wl.shape[0]*100

In [ ]:
feature_preprocess_pipeline.fit(df_wl)
df_wl = feature_preprocess_pipeline.transform(df_wl)

In [ ]:
df_wl[df_wl["bmi_field_imputed"].isnull()].shape[0]

In [ ]:
df_wl[df_wl["bmi_field_imputed"]>=30].shape[0]/df_wl.shape[0]*100

In [ ]:
df_wl[(df_wl["bmi_field"].isnull())&(df_wl["bmi_field_imputed"]>=30)].shape[0]/df_wl.shape[0]*100

# England

In [ ]:
df_en[df_en["bmi_field"].isnull()].shape[0]/df_en.shape[0] * 100

In [ ]:
df_en[df_en["bmi_field"]>=30].shape[0]/df_en.shape[0]*100

In [ ]:
feature_preprocess_pipeline.fit(df_en)
df_en = feature_preprocess_pipeline.transform(df_en)

In [ ]:
df_en[df_en["bmi_field_imputed"].isnull()].shape[0]

In [ ]:
df_en[df_en["bmi_field_imputed"]>=30].shape[0]/df_en.shape[0]*100

In [ ]:
df_en[(df_en["bmi_field"].isnull())&(df_en["bmi_field_imputed"]>=30)].shape[0]/df_en.shape[0]*100

# Unknown

In [ ]:
# for df_unknown use median (only 10 rows) and we dont' have IMD for multiple imputation

missing_pipeline_bmi_only = Pipeline([
    ('missing_imputer', internal_missing_pipeline_maker_bmi_only(cols_health= cols_health
                                               , create_new_col=True)),
])


feature_preprocess_pipeline_bmi_only = Pipeline([
   # ('outlier_pipeline', outlier_pipeline),
   ('missing_pipeline', missing_pipeline_bmi_only),
 #  ('discritiser_pipeline', discritiser_pipeline),
 #  ('feature_selection', feature_selection_pipeline)
])

In [ ]:
df_unkn[df_unkn["bmi_field"].isnull()].shape[0]

In [ ]:

feature_preprocess_pipeline_bmi_only.fit(df_unkn)
df_unkn = feature_preprocess_pipeline_bmi_only.transform(df_unkn)



In [ ]:
df_unkn[df_unkn["bmi_field_imputed"].isnull()].shape[0]

In [ ]:
df_unkn[df_unkn["traffic_intensity_field"].isnull()].shape[0]

In [ ]:
df_unkn[df_unkn["distance_nearest_road_field"].isnull()].shape[0]

In [ ]:
df_unkn[df_unkn["distance_major_road_field"].isnull()].shape[0]

In [ ]:
df_unkn['traffic_intensity_field'].median()

In [ ]:
df_unkn["traffic_intensity_field_imputed"] = df_unkn["traffic_intensity_field"]
df_unkn["distance_nearest_road_field_imputed"] = df_unkn["distance_nearest_road_field"]
df_unkn["distance_major_road_field_imputed"] = df_unkn["distance_major_road_field"]

df_unkn.loc[df_unkn["traffic_intensity_field"].isnull(), "traffic_intensity_field_imputed"] = df_unkn['traffic_intensity_field'].median()
df_unkn.loc[df_unkn["distance_nearest_road_field"].isnull(), "distance_nearest_road_field_imputed"] = df_unkn['distance_nearest_road_field'].median()
df_unkn.loc[df_unkn["distance_major_road_field"].isnull(), "distance_major_road_field_imputed"] = df_unkn['distance_major_road_field'].median()


In [ ]:
df_unkn[df_unkn["traffic_intensity_field_imputed"].isnull()].shape[0]

In [ ]:
df_unkn[df_unkn["distance_nearest_road_field_imputed"].isnull()].shape[0]

In [ ]:
df_unkn[df_unkn["distance_major_road_field_imputed"].isnull()].shape[0]

In [ ]:
# Re-Union
df_all = pd.concat([df_en, df_sct, df_wl, df_unkn])

In [ ]:
df_all.shape

In [ ]:
df_all[df_all["traffic_intensity_field_imputed"].isnull()].shape[0]

In [ ]:
df_all["bmi_field"].describe()

In [ ]:
df_all["bmi_field_imputed"].describe()

In [ ]:
#df_all[df_all["bmi_field"].isnull()][["bmi_field", "bmi_field_imputed"]]

In [ ]:
df_all['flag_asthma_at_cohort_start'].value_counts()

# Extra flags

Manke extra binary flags

In [ ]:
# y1y2 from recurrent file
df_y1y2_extra = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/csv/cohort_temp3_y1y2_gphesonly_raw_ukb_start.csv''')


In [ ]:
extra_flags = [x for x in df_y1y2_extra.columns if ( ((x.startswith("flag")) | (x.startswith("freq"))) & (x!="flag_asthma_at_cohort_start")  )]
extra_flags = ["eid"]+extra_flags


In [ ]:
df_all = pd.merge(df_all, df_y1y2_extra[extra_flags], on= "eid", how= "left")

In [ ]:
df_all['flag_asthma_at_cohort_start'].value_counts()

In [ ]:
df_all["traffic_intensity_field_imputed"].describe()

In [ ]:
df_all["distance_major_road_field_imputed"].describe()

In [ ]:
df_all["distance_nearest_road_field_imputed"].describe()

In [ ]:
df_all = make_binary_sex(df_all)
df_all = make_binary_age(df_all, 60)
df_all = make_binary_ethnicity(df_all)
df_all = make_binary_ethnicity_non_white(df_all)
df_all = make_categorical_exac_ocs_asthma(df_all)
df_all = make_categorical_exac_asthma(df_all)
df_all = make_categorical_non_exac_diag_asthma(df_all)
df_all = make_categorical_meds_all_asthma(df_all)
df_all = make_categorical_meds_other_asthma(df_all)
df_all = make_categorical_meds_ocs_asthma(df_all)
df_all = make_binary_cmrbd_prevalent_count(df_all)
df_all = make_binary_around_mean(df_all, "traffic_intensity_field_imputed", "traffic_high_bin_mean")
df_all = make_binary_around_median(df_all, "traffic_intensity_field_imputed", "traffic_high_bin_median")
df_all = make_binary_around_median(df_all, "distance_major_road_field_imputed", "distance_major_road_bin_median")
df_all = make_binary_around_mean(df_all, "distance_major_road_field_imputed", "distance_major_road_bin_mean")
df_all = make_binary_around_median(df_all, "distance_nearest_road_field_imputed", "distance_nearest_road_bin_median")
df_all = make_binary_around_mean(df_all, "distance_nearest_road_field_imputed", "distance_nearest_road_bin_mean")
df_all = make_binary_bmi_original(df_all)
df_all = make_binary_bmi_imputed(df_all)
df_all = make_binary_cmrbd_count(df_all)
df_all = make_binary_smoker_current(df_all)
df_all = make_binary_smoker_ever(df_all)
df_all = make_binary_smoker_previous(df_all)


In [ ]:
df_all["cmrbd_bin"].value_counts(dropna=False)

In [ ]:
df_all["bmi_30_original"].value_counts(dropna=False)

In [ ]:
df_all["bmi_30_imputed"].value_counts(dropna=False)

In [ ]:
df_all["bmi_field"].describe()

In [ ]:
df_all["bmi_field_imputed"].describe()

In [ ]:
df_all[(df_all["bmi_field"].notnull()) & (df_all["bmi_field"]!=df_all["bmi_field_imputed"])].shape

In [ ]:
df_all[df_all["bmi_field"].isnull()].shape

In [ ]:
df_all[df_all["bmi_field"].isnull()].shape[0]/df_all.shape[0]*100

In [ ]:
df_all[df_all["bmi_field_imputed"].isnull()].shape[0]/df_all.shape[0]*100

In [ ]:
df_all["cmrbd_bin_and_bmi"] = (df_all[["cmrbd_bin", "bmi_30_imputed"]].any(axis=1)).astype(int)

In [ ]:
df_all["cmrbd_bin_and_bmi"].value_counts(dropna=False)

In [ ]:
df_all['split'].value_counts()

In [ ]:
df_all = make_post_exac_ocs_yn_flags(df_all, n=3)
df_all = make_post_exac_ocs_yn_flags(df_all, n=2)

In [ ]:
df_all["flag_post_cohort_start_exac_y2"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_pre_cohort_start_meds_ocs_non_repeat_y2"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_post_cohort_start_exac_y1"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_post_cohort_start_exac_y2"].value_counts()

In [ ]:
df_all["flag_post_cohort_start_exac_ocs_y1"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_post_cohort_start_exac_ocs_y2"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_pre_cohort_start_exac"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["flag_pre_cohort_start_meds_ocs"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all['sex_female'].value_counts()

In [ ]:
df_all["age_60+"].value_counts()/df_all.shape[0]* 100

In [ ]:
df_all["freq_pre_exac_cat"].value_counts()

In [ ]:
df_all["freq_pre_meds_all_cat"].value_counts()

In [ ]:
df_all["freq_pre_meds_other_cat"].value_counts()

In [ ]:
df_all["freq_pre_meds_ocs_cat"].value_counts()

In [ ]:
df_all["eth_white"].value_counts()

In [ ]:
df_all["eth_non_white"].value_counts()

In [ ]:
df_all['traffic_high_bin_median'].value_counts()

In [ ]:
df_all['traffic_high_bin_mean'].value_counts()

In [ ]:
df_all['distance_major_road_bin_median'].value_counts()

In [ ]:
df_all['distance_major_road_bin_mean'].value_counts()

In [ ]:
df_all['distance_nearest_road_bin_median'].value_counts()

In [ ]:
df_all['distance_nearest_road_bin_mean'].value_counts()

In [ ]:
df_all['desc_smoking_at_baseline'].value_counts()

# Optional notebook based cmrbds bin (if exposure is any of the comorbidities)

In [ ]:
df_all['cmrbd_all_bin'] = df_all[['pheno_ht_pre_cohort_start', 
                              'pheno_diabetes_pre_cohort_start', 
                              'pheno_cvd_pre_cohort_start',
                              'pheno_ckd_pre_cohort_start',
                              'pheno_copd_pre_cohort_start',
                              'pheno_depression_pre_cohort_start',
                              'pheno_anxiety_pre_cohort_start', 
                              'pheno_motor_neuron_pre_cohort_start', 
                              'pheno_dementia_pre_cohort_start', 
                              'pheno_parkinson_pre_cohort_start', 'bmi_30_imputed']].any(axis=1).astype(int)

In [ ]:
df_all['cmrbd_sel_bin'] = df_all[['pheno_ht_pre_cohort_start', 
                              'pheno_diabetes_pre_cohort_start', 
                              'pheno_cvd_pre_cohort_start',
                              'pheno_ckd_pre_cohort_start',
                              'pheno_copd_pre_cohort_start',
                              'pheno_depression_pre_cohort_start',
                              'pheno_motor_neuron_pre_cohort_start', 
                              'pheno_dementia_pre_cohort_start', 
                              'pheno_parkinson_pre_cohort_start', 'bmi_30_imputed']].any(axis=1).astype(int)

In [ ]:
df_all["cmrbd_bin_and_bmi"].value_counts(dropna=False)

In [ ]:
df_all["cmrbd_all_bin"].value_counts(dropna=False)

In [ ]:
df_all.to_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''', index=False)